In [1]:
import os
import json
import google.generativeai as genai


GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")
print("set") if GOOGLE_API_KEY else print("unset")
genai.configure(api_key=GOOGLE_API_KEY)

set


/home/lukas/Programming/uni/threatintel/gemini/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from kscLLM.util import to_markdown, get_model
model = get_model()

# Load collected logs

In [6]:
# ../../pattern_matcher/resource/stix/observed/oss/positives_token_theft.json
with open("../tmp/logs.json", "r") as file:
    observables_bundle = json.load(file)

prompt = f"""Report malicious activity in the following STIX Domain Objects. Respond by showing the malicious SDOs and a short description of why they are malicious. 
A credential access attack is searched. First a token is fetched. This token is then extracted by an SSRF attack which calls the google.internal service accounts token endpoint.
Find for each step of this attack the observable.

Produce the response in the following format for each SDO:

Bundle ID: <bundle id>
Message: <message of the artifact>
Explanation: <explanation why this SDO is malicious>

<full SDO in json format>

{str(observables_bundle)}"""


response_indicator = model.generate_content(prompt)
# to_markdown(response_indicator.text)
print(response_indicator.text)

> ## STIX 2.1 Indicators from Provided Observables
> 
> These indicators are based on the provided STIX 2.1 bundles and aim to detect similar malicious activity:
> 
> **Indicator 1: Suspicious Service Account Token Fetching**
> 
> ```json
> {
>   "type": "indicator",
>   "id": "indicator--9b0a7471-886b-4a9e-916c-8302f886582d",
>   "spec_version": "2.1",
>   "created": "2024-09-09T19:21:04.000Z",
>   "modified": "2024-09-09T19:21:04.000Z",
>   "name": "Suspicious Service Account Token Fetching from Pod",
>   "description": "Detects attempts to fetch access tokens for service accounts from within a pod, potentially indicating unauthorized access attempts or privilege escalation.",
>   "pattern": "[kubernetes:pod:name = 'pacman/minimi-*' AND kubernetes:container_name = 'gke-metadata-server' AND event:log[string:message LIKE '%Fetching access token for service account%']]",
>   "pattern_type": "stix",
>   "pattern_version": "2.1",
>   "valid_from": "2024-09-09T19:21:04.000Z",
>   "kill_chain_phases": [
>     {
>       "kill_chain_name": "mitre-attack",
>       "phase_name": "credential-access"
>     },
>     {
>       "kill_chain_name": "mitre-attack",
>       "phase_name": "privilege-escalation"
>     }
>   ],
>   "indicator_types": [
>     "malicious-activity"
>   ],
>   "confidence": "80",
>   "severity": "medium"
> }
> ```
> 
> **Indicator 2: Successful New Entry Loading in Suspicious Pod**
> 
> ```json
> {
>   "type": "indicator",
>   "id": "indicator--4a148181-175a-4d3b-883d-734c8c73487e",
>   "spec_version": "2.1",
>   "created": "2024-09-09T19:21:04.000Z",
>   "modified": "2024-09-09T19:21:04.000Z",
>   "name": "Successful New Entry Loading in Suspicious Pod",
>   "description": "Detects successful loading of new entries in a pod potentially associated with malicious activity, indicating possible data exfiltration or command and control communication.",
>   "pattern": "[kubernetes:pod:name = 'pacman/minimi-*' AND kubernetes:container_name = 'gke-metadata-server' AND event:log[string:message LIKE '%Loading new entry succeeded%']]",
>   "pattern_type": "stix",
>   "pattern_version": "2.1",
>   "valid_from": "2024-09-09T19:21:04.000Z",
>   "kill_chain_phases": [
>     {
>       "kill_chain_name": "mitre-attack",
>       "phase_name": "command-and-control"
>     },
>     {
>       "kill_chain_name": "mitre-attack",
>       "phase_name": "exfiltration"
>     }
>   ],
>   "indicator_types": [
>     "malicious-activity"
>   ],
>   "confidence": "70",
>   "severity": "low"
> }
> ```
> 
> **Indicator 3: Suspicious HTTP Request to Metadata Server**
> 
> ```json
> {
>   "type": "indicator",
>   "id": "indicator--a73a0d56-6e4f-4a43-b99b-002d5c5a208a",
>   "spec_version": "2.1",
>   "created": "2024-09-09T19:21:04.000Z",
>   "modified": "2024-09-09T19:21:04.000Z",
>   "name": "Suspicious HTTP Request to Metadata Server for Service Account Token",
>   "description": "Detects suspicious HTTP requests targeting the Google metadata server aiming to retrieve service account tokens, often associated with SSRF attacks or attempts to gain unauthorized cloud resource access.",
>   "pattern": "[network-traffic:http_request:method = 'GET' AND network-traffic:http_request:uri LIKE '%/computeMetadata/v1/instance/service-accounts/default/token%' AND network-traffic:src_ref.value = '10.1.2.8']",
>   "pattern_type": "stix",
>   "pattern_version": "2.1",
>   "valid_from": "2024-09-09T19:21:04.000Z",
>   "kill_chain_phases": [
>     {
>       "kill_chain_name": "mitre-attack",
>       "phase_name": "reconnaissance"
>     },
>     {
>       "kill_chain_name": "mitre-attack",
>       "phase_name": "credential-access"
>     }
>   ],
>   "indicator_types": [
>     "malicious-activity"
>   ],
>   "confidence": "90",
>   "severity": "high"
> }
> ```
> 
> **Notes:**
> 
> * The `confidence` and `severity` levels are subjective and should be adjusted based on your specific environment and threat model.
> * The provided indicators are examples and may need to be adapted to your specific needs and detection capabilities.
> * Consider enriching these indicators with additional context, such as TTPs, known threat actors, and impacted platforms.
> * It's crucial to continuously monitor and update your indicators to stay ahead of evolving threats. 
